# 🧠 FunCAPTCHA Solver - Fine-Tuning trên Google Colab

Pipeline huấn luyện AI giải FunCAPTCHA với:

- **Classification**: Phân loại loại puzzle (rotate, match, select, shadow, pick, count)
- **Fine-Tuning**: Fine-tune Vision Transformer với LoRA
- **Angle Prediction**: Dự đoán góc xoay chính xác

---


## 1. Cài đặt môi trường


In [ ]:
# @title Cài đặt dependencies
!pip install -q torch torchvision transformers datasets accelerate peft
!pip install -q pillow opencv-python tqdm scikit-learn matplotlib seaborn
!pip install -q wandb albumentations pandas

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive & Clone Repo


In [ ]:
# @title Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/funcap/src')

!git clone https://github.com/your-username/funcap-solver.git /content/funcap 2>/dev/null || echo "Using local files"

## 3. Cấu hình & Hyperparameters


In [ ]:
# @title Cấu hình training
from pathlib import Path
from funcap_solver.config import Config, ModelConfig, TrainingConfig, DataConfig

config = Config()

# Model
config.model.classifier_backbone = "google/vit-base-patch16-224"
config.model.num_classes = 6
config.model.image_size = 224
config.model.rotation_bins = 360

# Training
config.training.cls_epochs = 20
config.training.cls_batch_size = 32
config.training.cls_learning_rate = 2e-4
config.training.use_lora = True
config.training.lora_r = 16
config.training.lora_alpha = 32
config.training.early_stopping_patience = 5
config.training.output_dir = Path("/content/drive/MyDrive/funcap_checkpoints")

# Data
config.data.class_labels = [
    "rotate_animal", "match_object", "select_tiles",
    "shadow_match", "pick_image", "count_objects"
]

print("Cấu hình đã sẵn sàng!")
print(f"  Backbone: {config.model.classifier_backbone}")
print(f"  Epochs: {config.training.cls_epochs}")
print(f"  Batch size: {config.training.cls_batch_size}")
print(f"  LoRA: {config.training.use_lora}")

## 4. Chuẩn bị Dataset

Upload dữ liệu FunCAPTCHA của bạn vào `/content/drive/MyDrive/funcap_data/` với cấu trúc:

```
funcap_data/
  train/
    image_001.png
    annotations.json
  val/
    ...
  test/
    ...
```

File `annotations.json` format:

```json
[
  {"image_path": "image_001.png", "class_label": 0, "angle": 45.0},
  ...
]
```


In [ ]:
# @title Tạo synthetic data mẫu (nếu chưa có dữ liệu thật)
import json
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

def generate_synthetic_data(output_dir: Path, num_samples: int = 500):
    """Tạo dữ liệu synthetic cho demo."""
    output_dir = Path(output_dir)
    for split in ["train", "val", "test"]:
        (output_dir / split).mkdir(parents=True, exist_ok=True)
    
    annotations = {"train": [], "val": [], "test": []}
    splits = ["train"] * 350 + ["val"] * 75 + ["test"] * 75
    
    for i in range(num_samples):
        split = splits[i] if i < len(splits) else "train"
        img = Image.new("RGB", (224, 224), (240, 240, 240))
        draw = ImageDraw.Draw(img)
        
        # Vẽ shape ngẫu nhiên
        shape_type = np.random.choice(["circle", "triangle", "square"])
        color = tuple(np.random.randint(50, 200, 3).tolist())
        cx, cy = np.random.randint(60, 164, 2)
        
        if shape_type == "circle":
            r = np.random.randint(20, 50)
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=color)
        elif shape_type == "triangle":
            s = np.random.randint(20, 50)
            pts = [(cx, cy-s), (cx-s, cy+s), (cx+s, cy+s)]
            draw.polygon(pts, fill=color)
        else:
            s = np.random.randint(20, 50)
            draw.rectangle([cx-s, cy-s, cx+s, cy+s], fill=color)
        
        class_label = np.random.randint(0, 6)
        angle = np.random.uniform(0, 360)
        
        # Xoay ảnh
        img = img.rotate(angle, resample=Image.BILINEAR)
        
        fname = f"sample_{i:05d}.png"
        img.save(output_dir / split / fname)
        annotations[split].append({
            "image_path": fname,
            "class_label": class_label,
            "angle": float(angle),
        })
    
    for split in ["train", "val", "test"]:
        with open(output_dir / split / "annotations.json", "w") as f:
            json.dump(annotations[split], f, indent=2)
    
    print(f"Generated {num_samples} samples")

DATA_DIR = Path("/content/funcap_data")
generate_synthetic_data(DATA_DIR, num_samples=500)

## 5. Fine-Tuning - Huấn luyện mô hình


In [ ]:
# @title Khởi tạo DataLoaders
from funcap_solver.data.dataset import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(
    DATA_DIR,
    batch_size=config.training.cls_batch_size,
    image_size=config.model.image_size,
    num_workers=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Test một batch
batch = next(iter(train_loader))
print(f"Batch keys: {list(batch.keys())}")
print(f"Pixel values shape: {batch['pixel_values'].shape}")
print(f"Labels shape: {batch['class_label'].shape}")

In [ ]:
# @title Train Combined Model (Classification + Angle)
import wandb
from funcap_solver.models.model import FunCaptchaModel
from funcap_solver.training.trainer import Trainer

# Login W&B (optional)
# wandb.login()
# wandb.init(project="funcap-solver", config=vars(config.training))

# Khởi tạo model
model = FunCaptchaModel(
    backbone_name=config.model.classifier_backbone,
    num_classes=config.model.num_classes,
    num_angle_bins=config.model.rotation_bins,
    dropout=config.model.classifier_dropout,
    use_lora=config.training.use_lora,
    lora_r=config.training.lora_r,
    lora_alpha=config.training.lora_alpha,
)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# @title Bắt đầu Training
from pathlib import Path

trainer = Trainer(model, config.training, use_wandb=False)

history = trainer.fit(
    train_loader,
    val_loader,
    task="combined",
    checkpoint_dir=config.training.output_dir,
)

# Plot
trainer.plot_history(history)

## 6. Đánh giá


In [ ]:
# @title Evaluate on test set
test_metrics = trainer.evaluate(test_loader, task="combined")
print("\n📊 Test Results:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# @title Test inference
from PIL import Image
from funcap_solver.inference.solver import FunCaptchaSolver

# Save model checkpoint first
import torch
ckpt_path = config.training.output_dir / "best_model.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "epoch": config.training.cls_epochs,
}, ckpt_path)

# Load solver
solver = FunCaptchaSolver(ckpt_path, config)

# Test on a sample
test_img = Image.open(DATA_DIR / "test" / "sample_00450.png").convert("RGB")
result = solver.solve(test_img)
print(f"\n🔍 Prediction:")
print(f"  Puzzle type:    {result['puzzle_type']}")
print(f"  Angle:          {result['angle']:.1f}°")
print(f"  Cls confidence: {result['cls_confidence']:.4f}")
print(f"  Angle confidence: {result['angle_confidence']:.4f}")

## 7. Export Model


In [ ]:
# @title Save to Drive & Download
!cp -r {config.training.output_dir} /content/drive/MyDrive/funcap_checkpoints
print(f"✅ Checkpoint saved to Drive: /content/drive/MyDrive/funcap_checkpoints")

# Zip for download
!zip -r /content/funcap_model.zip {config.training.output_dir}
from google.colab import files
# files.download("/content/funcap_model.zip")

---

## 🎯 Tổng kết

Pipeline này bao gồm:

1. ✅ **Data Collection** - Thu thập & synthetic data generation
2. ✅ **Classification** - Phân loại 6 loại FunCAPTCHA puzzle
3. ✅ **Rotation Prediction** - Dự đoán góc xoay 360°
4. ✅ **LoRA Fine-tuning** - Parameter-efficient training
5. ✅ **Inference API** - Giải CAPTCHA end-to-end
